# AdaBoost 如何聚焦难样本？

**面试回答：**AdaBoost 每轮提升错分样本权重，用弱分类器误差决定其投票权；错标点也会被持续放大，因此需监控样本权重。

## 真实案例

退款审核按订单金额训练阈值桩树，标签中有一条人工错标，观察其权重。

In [1]:
import numpy as np  # 导入 NumPy 手写 AdaBoost。
order=np.array(['A01','A02','A03','A04','A05','A06','A07','A08'])  # 构造订单编号。
x=np.array([20.,35.,50.,65.,80.,95.,110.,125.])  # 记录订单金额。
y=np.array([-1,-1,-1,1,1,1,-1,1])  # 记录退款标签，其中 A07 为错标反例。
print('订单 | 金额 | 退款标签')  # 输出订单表头。
for n,v,c in zip(order,x,y):  # 展示业务样本。
    print(n,v,int(c))  # 输出一条订单。

订单 | 金额 | 退款标签
A01 20.0 -1
A02 35.0 -1
A03 50.0 -1
A04 65.0 1
A05 80.0 1
A06 95.0 1
A07 110.0 -1
A08 125.0 1


## Baseline / 基线

基线使用一个固定 60 元阈值。

In [2]:
base=np.where(x>60,1,-1)  # 用固定阈值预测。
base_acc=float(np.mean(base==y))  # 计算基线准确率。
print('固定阈值准确率=',base_acc)  # 输出基线。

固定阈值准确率= 0.875


In [3]:
weight=np.full(len(x),1/len(x))  # 均匀初始化样本权重。
score=np.zeros(len(x))  # 初始化集成得分。
history=[]  # 保存每轮错误率和最大权重。
for round_id in range(3):  # 训练三个弱学习器。
    candidates=np.unique(x)[:-1]  # 生成阈值候选。
    errors=[np.sum(weight*(np.where(x>t,1,-1)!=y)) for t in candidates]  # 计算每个树桩的加权错误率。
    threshold=candidates[int(np.argmin(errors))]  # 选择加权错误最小阈值。
    pred=np.where(x>threshold,1,-1)  # 生成当前弱学习器预测。
    error=np.sum(weight*(pred!=y))  # 计算当前加权错误率。
    alpha=.5*np.log((1-error)/(error+1e-9))  # 计算弱学习器投票权。
    weight*=np.exp(-alpha*y*pred)  # 放大错分样本权重。
    weight/=weight.sum()  # 归一化样本权重。
    score+=alpha*pred  # 累积集成得分。
    history.append((threshold,error,alpha,weight.copy()))  # 保存中间状态。
final=np.where(score>=0,1,-1)  # 将集成得分转成最终类别。
acc=float(np.mean(final==y))  # 计算集成准确率。
print('每轮阈值/误差/alpha:',[(round(t,1),round(e,3),round(a,3)) for t,e,a,w in history])  # 输出关键中间量。
print('末轮样本权重:',np.round(history[-1][3],3))  # 输出难样本权重。

每轮阈值/误差/alpha: [(50.0, 0.125, 0.973), (110.0, 0.214, 0.65), (50.0, 0.318, 0.381)]
末轮样本权重: [0.033 0.033 0.033 0.122 0.122 0.122 0.5   0.033]


## 结果解读

错分样本获得更高权重，下一轮树会优先解释它；若该点是错标，这种机制会伤害泛化。

In [4]:
print('订单 | 标签 | 最终得分 | 权重')  # 输出诊断表头。
for n,c,s,w in zip(order,y,score,history[-1][3]):  # 展示各样本关注程度。
    print(n,int(c),round(float(s),3),round(float(w),3))  # 输出订单分数和权重。
print('生产差距：需标签审计、权重上限、时间验证与概率校准。')  # 说明工程边界。

订单 | 标签 | 最终得分 | 权重
A01 -1 -2.004 0.033
A02 -1 -2.004 0.033
A03 -1 -2.004 0.033
A04 1 0.704 0.122
A05 1 0.704 0.122
A06 1 0.704 0.122
A07 -1 0.704 0.5
A08 1 2.004 0.033
生产差距：需标签审计、权重上限、时间验证与概率校准。


## 失败案例与修复

故意的 A07 错标会成为最大权重样本；修复是复核标签或限制单样本权重，而不是继续加轮数。

In [5]:
max_index=int(history[-1][3].argmax())  # 找到最终权重最大样本。
print('失败：最大权重订单=',order[max_index])  # 输出被持续关注的异常点。
clipped=np.minimum(history[-1][3],.3)  # 演示对单样本权重做上限。
clipped/=clipped.sum()  # 重新归一化截断权重。
print('修复：截断后最大权重=',round(float(clipped.max()),3))  # 输出风险控制效果。
print('权重截断是护栏，根因仍需回查标签。')  # 说明正确治理顺序。

失败：最大权重订单= A07
修复：截断后最大权重= 0.375
权重截断是护栏，根因仍需回查标签。


In [6]:
assert len(order)>=5  # 保护样本数。
assert acc>=base_acc  # 保护集成不弱于基线。
assert history[-1][3].sum().round()==1  # 保护权重归一化。
assert order[max_index]=='A07'  # 保护错标被放大的失败现象。